In [8]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

data = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

In [9]:
!curl -o hw_data.csv $data

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  272k  100  272k    0     0   3676      0  0:01:15  0:01:15 --:--:-- 61916     0    0     0      0      0 --:--:--  0:00:17 --:--:--     0 0     0    0     0      0      0 --:--:--  0:00:43 --:--:--     00    0     0    0     0      0      0 --:--:--  0:00:51 --:--:--     0


In [11]:
df = pd.read_csv('hw_data.csv')

In [12]:
print(df.dtypes, '\n\n\n', df.isnull().sum())

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object 


 lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [13]:
df.isnull().any()

lead_source                  True
industry                     True
employment_status            True
location                     True
annual_income                True
number_of_courses_viewed    False
interaction_count           False
lead_score                   True
converted                   False
dtype: bool

In [27]:
str_col = list(df.select_dtypes(include=['object', 'string']).columns)
num_col = list(df.select_dtypes(include=['number']).columns)

num_col = [c for c in num_col if c != 'converted']
str_col = [c for c in str_col if c != 'converted']

na_col = list(df[str_col].columns[df[str_col].isnull().any()])
na_num_col = list(df[num_col].columns[df[num_col].isnull().any()])

df[na_num_col] = df[na_num_col].fillna(0) # In parallel => faster than loops
df[na_col] = df[na_col].fillna('NA')

In [28]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NA,0.0,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### Q1. What is the most frequent observation (mode) for the column industry?

In [29]:
df.industry.mode()[0]

'technology'

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.
### Q2. What are the two features that have the biggest correlation?

In [30]:
corr_matrix = df[num_col].corr()
print("\nCorrelation Matrix:")
print(corr_matrix)

# Finding pair with highest absolute correlation
corr_unstacked = corr_matrix.abs().unstack()
corr_pairs = corr_unstacked[corr_unstacked < 1.0].sort_values(ascending=False)
q2_answer = corr_pairs.index[0]
print()
print("Answer: ", q2_answer)


Correlation Matrix:
                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  

Answer:  ('interaction_count', 'lead_score')


**Split data**

In [31]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

### Q3. Which variables has the biggest mutual information score?

In [32]:
def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[str_col].apply(calculate_mi).sort_values(ascending=False)
print("\nMutual Information Scores:")
print(mi_scores)

mi_scores.index[0]


Mutual Information Scores:
lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64


'lead_source'

### Q4. Accuracy of Logistic Regression

In [33]:
all_features = str_col + num_col

dv = DictVectorizer(sparse=False)
train_dict = df_train[all_features].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[all_features].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
original_accuracy = accuracy_score(y_val, y_pred)

print(f"Accuracy: {round(original_accuracy, 2)}")

Accuracy: 0.65


### Q5. Which feature has the smallest difference?

In [34]:
diffs = {}

for feature in all_features:
    subset_features = [f for f in all_features if f != feature]
    
    dv_sub = DictVectorizer(sparse=False)
    X_tr_sub = dv_sub.fit_transform(df_train[subset_features].to_dict(orient='records'))
    X_va_sub = dv_sub.transform(df_val[subset_features].to_dict(orient='records'))
    
    model_sub = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model_sub.fit(X_tr_sub, y_train)
    
    acc_sub = accuracy_score(y_val, model_sub.predict(X_va_sub))
    diffs[feature] = original_accuracy - acc_sub

diffs_df = pd.DataFrame(list(diffs.items()), columns=['feature', 'accuracy_diff']).sort_values('accuracy_diff')
print("\nFeature Elimination Differences (original - without feature):")
print(diffs_df)

q5_answer = diffs_df.iloc[0]['feature']
print(f"\nQ5 Answer (Smallest difference): {q5_answer}")


Feature Elimination Differences (original - without feature):
                    feature  accuracy_diff
4             annual_income         -0.079
1                  industry          0.000
2         employment_status          0.000
3                  location          0.000
7                lead_score          0.001
5  number_of_courses_viewed          0.002
0               lead_source          0.003
6         interaction_count          0.044

Q5 Answer (Smallest difference): annual_income


### Q6. Which of these C leads to the best accuracy on the validation set?

In [36]:
c_values = [0.000001, 0.00001, 0.0001, 0.001]
c_results = {}

for c in c_values:
    model_c = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model_c.fit(X_train, y_train)
    
    y_pred_c = model_c.predict(X_val)
    acc_c = accuracy_score(y_val, y_pred_c)
    c_results[c] = round(acc_c, 3)

print("\nC parameter search results:")
for c, acc in c_results.items():
    print(f"C={c}: Accuracy={acc}")

best_c = max(c_results, key=c_results.get)
print(f"\nBest C parameter: {best_c}")


C parameter search results:
C=1e-06: Accuracy=0.598
C=1e-05: Accuracy=0.598
C=0.0001: Accuracy=0.613
C=0.001: Accuracy=0.645

Best C parameter: 0.001
